# Chapter 03 — Discovery, Use-Case Scoping and Value Case · Practice

**Goal:** take one customer request ("we need AI for claims") through every scoping step in the chapter: an assumptions log, interview notes structured by Claude **and** OpenAI through one small interface, a workflow walk, gates and scoring, gain classification, a problem statement, requirement tracing, a first-cut solution shape, a value case on a baseline, and a go/no-go rule. The mini-project repeats it for a second client and prints a one-page scope brief.

- Theory: [README.md](README.md) (read section 3.x before running the matching section here)
- Estimated time: 2–3 hours
- Estimated API cost: **$0** by default. The two optional live cells cost about **$0.01–0.02** in total on `claude-sonnet-5-5` and `gpt-6-luna`.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`; `.env` only if you go live), structured outputs from Claude [02](../../anthropic-claude/02-tool-use/README.md) or OpenAI [02](../../openai-codex/02-function-calling-and-structured-outputs/README.md)

| Marker | Meaning |
|---|---|
| **(offline)** | Runs with no key. Model calls go to `FakeLLM`: the **real** `anthropic.Anthropic` and `openai.OpenAI` clients with their HTTP transport swapped for scripted replies, so `messages.parse()` and `responses.parse()` run for real. |
| **(live, costs money)** | Calls the real APIs. Runs only when `RUN_LIVE=1` is set in `.env` together with both API keys; otherwise it prints "Skipped". |

Most of this chapter is plain Python on purpose: scoping is arithmetic and judgment, and only one step (structuring interview notes) uses a model.

In [ ]:
import json
import os
import re
from dataclasses import dataclass
from itertools import count
from typing import Literal

import anthropic
import httpx2
import openai
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
RUN_LIVE = (os.getenv("RUN_LIVE") == "1" and bool(os.getenv("ANTHROPIC_API_KEY"))
            and bool(os.getenv("OPENAI_API_KEY")))

print("anthropic", anthropic.__version__, "| openai", openai.__version__,
      "| models:", CLAUDE_MODEL, OPENAI_MODEL, "| RUN_LIVE =", RUN_LIVE)

### Offline toolkit (run this once)

`FakeLLM(responder)` builds both SDK clients on one `httpx2.MockTransport`. `responder(path, body)` returns the JSON object the "model" should answer with; the fake wraps it in a Messages API body (`/v1/messages`) or a Responses API body (`/v1/responses`). `.requests` records what each SDK sent.

`structured()` is the **one small interface** the chapter uses for both vendors. It trusts a parsed result only when the model finished normally (`stop_reason == "end_turn"` on Claude, `status == "completed"` on OpenAI).

In [ ]:
_ids = count(1)


class FakeLLM:
    def __init__(self, responder):
        self.responder, self.requests = responder, []
        transport = httpx2.MockTransport(self._handle)
        self.clients = {
            "claude": anthropic.Anthropic(api_key="offline", max_retries=0,
                                          http_client=anthropic.DefaultHttpxClient(transport=transport)),
            "openai": openai.OpenAI(api_key="offline", max_retries=0,
                                    http_client=openai.DefaultHttpxClient(transport=transport)),
        }

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        path, body = request.url.path, json.loads(request.content)
        self.requests.append((path, body))
        text, n = json.dumps(self.responder(path, body)), next(_ids)
        if path.endswith("/messages"):
            out = {"id": f"msg_{n}", "type": "message", "role": "assistant", "model": body["model"],
                   "content": [{"type": "text", "text": text}], "stop_reason": "end_turn",
                   "stop_sequence": None, "usage": {"input_tokens": 0, "output_tokens": 0}}
        else:
            out = {"id": f"resp_{n}", "object": "response", "created_at": 0, "status": "completed",
                   "model": body["model"], "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
                   "output": [{"type": "message", "id": f"msg_{n}", "role": "assistant", "status": "completed",
                               "content": [{"type": "output_text", "text": text, "annotations": []}]}]}
        return httpx2.Response(200, json=out)


def request_text(path: str, body: dict) -> str:
    # The user text of a request, for either vendor.
    if path.endswith("/messages"):
        return body["messages"][-1]["content"]
    return body["input"] if isinstance(body["input"], str) else json.dumps(body["input"])


def structured(vendor: str, system: str, user: str, schema: type[BaseModel], clients: dict):
    # One interface, two vendors: returns a validated `schema` instance or raises.
    if vendor == "claude":
        r = clients["claude"].messages.parse(
            model=CLAUDE_MODEL, max_tokens=2000, system=system,
            messages=[{"role": "user", "content": user}], output_format=schema)
        if r.stop_reason != "end_turn" or r.parsed_output is None:
            raise RuntimeError(f"claude: stop_reason={r.stop_reason}")
        return r.parsed_output
    r = clients["openai"].responses.parse(model=OPENAI_MODEL, instructions=system,
                                          input=user, text_format=schema)
    if r.status != "completed" or r.output_parsed is None:
        raise RuntimeError(f"openai: status={r.status}")
    return r.output_parsed


print("offline toolkit ready")

### The case: Harbor Mutual (fictional)

A home insurer receives about 4,000 loss emails a week. The COO asked for "AI for claims". Below is the interview with the claims intake lead and the workflow you walked with three adjusters. All names and numbers are invented for this course.

In [ ]:
TRANSCRIPT = '''Architect: Walk me through the last loss email you handled.
Dana (claims intake lead): This morning a customer wrote that a pipe burst in the kitchen on Sunday. No policy number,
just a name and street address. I looked the policy up by address, picked water damage, typed the details into
ClaimDesk and opened the claim. About 20 minutes, because of the lookup.
Architect: How many of these come in?
Dana: Our export says 4,112 last week. After the March storm it was over 9,000.
Architect: How long does a typical one take?
Dana: Our time study in May said 18 minutes on average. Lookups are the slow part.
Architect: What goes wrong?
Dana: About 6% of claims get reopened because something was typed wrong, mostly the loss date.
Architect: What would success look like?
Dana: Claims opened the same hour. Right now the median is 26 hours. Honestly I think most adjusters would love it.
Architect: Anything we must not do?
Dana: Anything over 25 thousand dollars needs an adjuster's approval. And customer data stays in the US.'''

WEEKLY_EMAILS = 4112
print(len(TRANSCRIPT.split()), "words in the transcript")

## 3.1 The discovery arc and owning ambiguity

You will not have complete information, so make the unknowns explicit. An **assumptions log** lists what you believe, what breaks if it is wrong, how you will test it, who owns the test and by when. Test the riskiest first. [Theory](README.md#31-the-discovery-arc-and-owning-ambiguity)

In [ ]:
@dataclass
class Assumption:
    text: str
    impact: int          # 1-5: how badly the plan breaks if this is wrong
    uncertainty: int     # 1-5: how unsure we are today
    test: str
    owner: str
    due_day: int

    @property
    def risk(self) -> int:
        return self.impact * self.uncertainty


LOG = [
    Assumption("Emails contain the policy number 80%+ of the time", 3, 4, "Label 200 historical emails", "Claims ops lead", 5),
    Assumption("ClaimDesk has a create-draft API", 5, 3, "Read the API docs, call the sandbox", "IT architect", 3),
    Assumption("Adjusters will review drafts instead of retyping them", 5, 4, "Shadow 3 adjusters on a mock-up", "Architect", 8),
    Assumption("Storm peaks stay under 10,000 emails a week", 2, 2, "Check 3 years of exports", "Claims ops lead", 6),
]

for a in sorted(LOG, key=lambda a: (-a.risk, a.due_day)):
    print(f"risk {a.risk:2}  day {a.due_day}  {a.owner:16} {a.text}  ->  {a.test}")

> **Observe:** the adoption assumption ranks first. If adjusters retype drafts, there is no time saved and no value case, so it gets tested in week one, before any model work.

## 3.2 Structured discovery with stakeholders

A model can turn interview notes into structured facts, but it must not invent them. The schema asks for the speaker's **quote** and a `confirmed` flag; then a **code gate** checks that every quote really appears in the transcript. The same request goes to Claude and to OpenAI through `structured()`. [Theory](README.md#32-structured-discovery-with-stakeholders)

In [ ]:
class Fact(BaseModel):
    topic: Literal["goal", "workflow", "volume", "pain", "data", "risk", "constraint", "success"]
    statement: str
    quote: str            # the stakeholder's own words
    confirmed: bool       # False = an assumption to verify


class DiscoveryNotes(BaseModel):
    facts: list[Fact]
    open_questions: list[str]


NOTES_SYSTEM = ("Turn the interview transcript into facts. For each fact, copy the speaker's exact words into `quote`. "
                "Set confirmed=true only when the speaker gives a number with a source (an export, a study, a policy). "
                "List what is still unknown as open questions. Do not add facts that are not in the transcript.")


def f(topic, statement, quote, confirmed):
    return {"topic": topic, "statement": statement, "quote": quote, "confirmed": confirmed}


# What the scripted "model" answers. One fact is deliberately unsupported, to show the gate below.
HARBOR_NOTES = {"facts": [
    f("volume", "About 4,100 loss emails a week", "Our export says 4,112 last week", True),
    f("volume", "Storm peaks above 9,000 a week", "After the March storm it was over 9,000", True),
    f("workflow", "18 minutes per email on average", "Our time study in May said 18 minutes on average", True),
    f("pain", "6% of claims reopened for typing errors", "About 6% of claims get reopened", False),
    f("success", "Median email-to-claim time is 26 hours; goal is the same hour", "Right now the median is 26 hours", True),
    f("success", "Adjusters want the tool", "most adjusters would love it", False),
    f("constraint", "Claims over $25k need adjuster approval", "Anything over 25 thousand dollars needs an adjuster's approval", True),
    f("constraint", "Customer data must stay in the US", "customer data stays in the US", True),
    f("success", "Adoption will reach 90%", "Adoption will be at least 90 percent", True),
], "open_questions": ["Source of the 6% reopen rate?", "Does ClaimDesk have an API?", "Share of emails with a policy number?"]}

fake = FakeLLM(lambda path, body: HARBOR_NOTES)


def unsupported(notes: DiscoveryNotes, transcript: str) -> list[Fact]:
    # The code gate: a fact whose quote is not in the transcript is not evidence.
    flat = " ".join(transcript.split())
    return [x for x in notes.facts if " ".join(x.quote.split()) not in flat]


for vendor in ("claude", "openai"):
    notes = structured(vendor, NOTES_SYSTEM, TRANSCRIPT, DiscoveryNotes, fake.clients)
    bad = unsupported(notes, TRANSCRIPT)
    print(f"{vendor}: {len(notes.facts)} facts, {sum(x.confirmed for x in notes.facts)} confirmed, "
          f"unsupported: {[x.statement for x in bad]}")

kept = [x for x in notes.facts if x not in bad]
print("\nassumptions to add to the log:", [x.statement for x in kept if not x.confirmed])

In [ ]:
# What each SDK actually sent: same schema, different wire format.
for path, body in fake.requests[-2:]:
    fmt = body.get("output_config", {}).get("format") or body.get("text", {}).get("format")
    print(path, "| system field:", "system" if "system" in body else "instructions",
          "| format type:", fmt["type"], "| required fields:", fmt["schema"]["required"])

> **Observe:** both vendors return the same typed `DiscoveryNotes`, and the gate drops the invented "90% adoption" fact whatever the model's `confirmed` flag says. "Most adjusters would love it" survives as an **unconfirmed** opinion: it belongs in the assumptions log, not in the value case.

In [ ]:
# (live, costs money: about $0.01 for both calls) the same extraction on the real APIs
if not RUN_LIVE:
    print("Skipped: set RUN_LIVE=1 plus ANTHROPIC_API_KEY and OPENAI_API_KEY in .env to run this cell.")
else:
    live = {"claude": anthropic.Anthropic(), "openai": openai.OpenAI()}
    for vendor in ("claude", "openai"):
        notes = structured(vendor, NOTES_SYSTEM, TRANSCRIPT, DiscoveryNotes, live)
        print(f"{vendor}: {len(notes.facts)} facts, unsupported quotes: {len(unsupported(notes, TRANSCRIPT))}")
        for x in notes.facts:
            print(f"   [{'C' if x.confirmed else '?'}] {x.topic:10} {x.statement}")

## 3.3 Finding AI opportunities in a real workflow

Walk one workflow step by step, record minutes and share of items per step, then tag each step: rule → code, language judgment → LLM candidate, accountable decision → human-only, already automated → leave it. [Theory](README.md#33-finding-ai-opportunities-in-a-real-workflow)

In [ ]:
@dataclass
class Step:
    name: str
    minutes: float               # per item that goes through the step
    share: float                 # share of items that go through it
    logic: Literal["rule", "judgment", "accountable"]
    unstructured_input: bool
    automated: bool = False


WORKFLOW = [
    Step("Read the email and classify the loss", 4, 1.0, "judgment", True),
    Step("Find the policy by number or address", 6, 1.0, "rule", False),
    Step("Extract loss date, address, description, contact", 5, 1.0, "judgment", True),
    Step("Check for duplicate claims", 1, 1.0, "rule", False, automated=True),
    Step("Type the fields into ClaimDesk and open the claim", 2, 1.0, "rule", False),
    Step("Approve claims over $25k", 6, 0.08, "accountable", False),
]


def tag(step: Step) -> str:
    if step.automated:
        return "already automated"
    if step.logic == "accountable":
        return "human-only (support with a draft)"
    if step.logic == "rule":
        return "code or tool"
    return "LLM candidate" if step.unstructured_input else "review by hand"


by_tag: dict[str, float] = {}
for s in WORKFLOW:
    hours = s.minutes * s.share * WEEKLY_EMAILS / 60
    by_tag[tag(s)] = by_tag.get(tag(s), 0) + hours
    print(f"{tag(s):34} {hours:6.0f} h/week  {s.name}")
print({k: round(v) for k, v in by_tag.items()})

> **Observe:** the LLM candidates (classify + extract) are about 600 hours a week, but the "code or tool" steps (policy lookup, typing into ClaimDesk) are about 550. Half of the value comes from plain integration, which a good scope includes and a demo-first project misses.

## 3.4 Qualifying and prioritizing use cases

Hard gates first, then a weighted value and feasibility score, then a quadrant. The function is the README's `prioritize`. [Theory](README.md#34-qualifying-and-prioritizing-use-cases)

In [ ]:
GATES = ("data_accessible", "owner", "measurable", "errors_reversible")


def prioritize(uc: dict) -> tuple[str, float, float]:
    failed = [g for g in GATES if not uc[g]]
    if failed:
        return ("park: " + ", ".join(failed), 0.0, 0.0)
    value = 0.4 * uc["pain_size"] + 0.3 * uc["strategic_fit"] + 0.3 * uc["volume"]
    feas = (0.3 * uc["data_readiness"] + 0.3 * uc["complexity_fit"]
            + 0.2 * uc["integration_ease"] + 0.2 * uc["team_readiness"])
    if value >= 3.5 and feas >= 3.5:
        quadrant = "quick win: pilot first"
    elif value >= 3.5:
        quadrant = "strategic bet: plan and fund"
    elif feas >= 3.5:
        quadrant = "self-service: enable, do not build"
    else:
        quadrant = "park"
    return (quadrant, round(value, 2), round(feas, 2))


def uc(name, gates=(1, 1, 1, 1), value=(3, 3, 3), feas=(3, 3, 3, 3)):
    return dict(name=name, **dict(zip(GATES, map(bool, gates))),
                **dict(zip(("pain_size", "strategic_fit", "volume"), value)),
                **dict(zip(("data_readiness", "complexity_fit", "integration_ease", "team_readiness"), feas)))


USE_CASES = [
    uc("Intake triage and extraction", value=(5, 5, 5), feas=(4, 4, 3, 4)),
    uc("Adjuster letter drafting", value=(3, 2, 4), feas=(4, 4, 4, 4)),
    uc("Policy Q&A for call-center agents", value=(4, 4, 4), feas=(2, 4, 3, 3)),
    uc("LLM fraud-ring detection", gates=(0, 1, 1, 0), value=(5, 4, 3), feas=(1, 2, 2, 2)),
    uc("'AI for everything' assistant for the CEO", gates=(1, 0, 0, 1), value=(2, 3, 1), feas=(3, 2, 3, 3)),
]

for u in sorted(USE_CASES, key=lambda u: -prioritize(u)[1]):
    quadrant, v, fz = prioritize(u)
    print(f"value {v:4}  feas {fz:4}  {quadrant:36} {u['name']}")

> **Try it:** the security team says the policy documents for the Q&A use case will be indexed next quarter. Raise its `data_readiness` to 4 and rerun. It becomes a second quick win, which is how a strategic bet gets promoted once a blocker clears.

## 3.5 Value pillars: structural vs marginal gain

Compare two designs for the same use case. A **marginal** design keeps every human step and makes some of them faster. A **structural** design removes human steps from the unit of work. [Theory](README.md#35-value-pillars-structural-vs-marginal-gain)

In [ ]:
@dataclass
class Design:
    name: str
    human_steps: int          # steps a person performs per item
    minutes: float            # person-minutes per item
    pillar: str
    metric: str


def classify_gain(before: Design, after: Design) -> str:
    if after.human_steps < before.human_steps:
        return "structural"
    if after.minutes < before.minutes:
        return "marginal"
    return "none"


TODAY = Design("Today", 5, 18, "-", "-")
DESIGNS = [
    Design("AI summary shown next to the email", 5, 14, "efficiency", "minutes per claim"),
    Design("Auto-drafted claim, adjuster reviews exceptions", 1, 6, "transformation",
           "email-to-claim time, claims per adjuster per day"),
]
for d in DESIGNS:
    freed = (TODAY.minutes - d.minutes) * WEEKLY_EMAILS / 60
    print(f"{classify_gain(TODAY, d):10} {freed:5.0f} h/week freed  pillar={d.pillar:14} prove it with: {d.metric}  <- {d.name}")

> **Observe:** the summary design frees about 270 hours a week spread over 42 people (about 6 hours each), which is hard to see in 30 days of noisy data. The auto-draft design changes the unit of work (one review step instead of five) and shows up directly in claim timestamps.

## 3.6 The problem statement

A small lint catches the two common faults: solution words in the problem, and a baseline or target with no number. [Theory](README.md#36-the-problem-statement)

In [ ]:
class ProblemStatement(BaseModel):
    who: str
    job: str
    baseline: str
    target: str
    constraints: list[str]
    out_of_scope: list[str]
    decision_by: str


SOLUTION_WORDS = ("chatbot", "agent", "gpt", "llm", "claude", "copilot", "ai-powered")


def lint(ps: ProblemStatement) -> list[str]:
    text = " ".join([ps.who, ps.job, ps.baseline, ps.target]).lower()
    issues = [f"solution word '{w}' in the problem" for w in SOLUTION_WORDS if re.search(rf"\b{w}\b", text)]
    if not re.search(r"\d", ps.baseline):
        issues.append("baseline has no number")
    if not re.search(r"\d", ps.target):
        issues.append("target has no number")
    if not ps.out_of_scope:
        issues.append("nothing is out of scope")
    if not ps.decision_by.strip():
        issues.append("no decision date or rule")
    return issues


draft = ProblemStatement(who="Claims team", job="Use a GPT-powered agent to handle claims emails",
                         baseline="Slow and error-prone", target="Much faster", constraints=[], out_of_scope=[],
                         decision_by="")
HARBOR_PS = ProblemStatement(
    who="Claims intake adjusters (42 people, 2 sites)",
    job="Turn a loss email into an opened, correctly classified claim",
    baseline="18 min per email (May time study); median 26 h email-to-claim; 6% reopened for wrong data",
    target="Median under 2 h; reopen rate at or below 6%; no extra headcount",
    constraints=["US data only", "PII", "Existing ClaimDesk system", "Adjuster approves every claim over $25k"],
    out_of_scope=["Fraud scoring", "Payments", "Customer replies (phase 2)"],
    decision_by="Day 30 of the pilot, on the agreed success criteria")
print("draft:", lint(draft))
print("final:", lint(HARBOR_PS) or "clean")

## 3.7 From business requirements to functional and infrastructure requirements

Each business requirement (BR) must trace to at least one functional requirement (FR), and the infrastructure requirements (IR) must cover the categories a security and platform review will ask about. [Theory](README.md#37-from-business-requirements-to-functional-and-infrastructure-requirements)

In [ ]:
REQUIRED_IR = {"throughput", "latency", "integration", "identity", "residency", "retention"}

BRS = {"BR1": "Claims open within 1 business hour", "BR2": "Fewer claims reopened for wrong data",
       "BR3": "Customer data stays in the US and is auditable"}
FRS = {"FR1": ("Classify the loss type", ["BR1"]),
       "FR2": ("Extract policy number, loss date, address, contact", ["BR1", "BR2"]),
       "FR3": ("Create a draft claim in ClaimDesk", ["BR1"]),
       "FR4": ("Validate fields against the policy record; route mismatches to an adjuster", ["BR2"])}
IRS = {"IR1": ("throughput", "Peak 150 emails/hour after storms"),
       "IR2": ("latency", "p95 under 90 s per email"),
       "IR3": ("integration", "ClaimDesk REST API with a service account"),
       "IR4": ("identity", "SSO for reviewers")}


def trace(brs: dict, frs: dict, irs: dict) -> dict:
    covered = {br for _, targets in frs.values() for br in targets}
    return {"br_without_fr": sorted(set(brs) - covered),
            "missing_ir_categories": sorted(REQUIRED_IR - {cat for cat, _ in irs.values()})}


print("first draft:", trace(BRS, FRS, IRS))
FRS["FR5"] = ("Log inputs, outputs and the reviewer of every claim", ["BR3"])
IRS["IR5"] = ("residency", "US-region processing on the chosen platform")
IRS["IR6"] = ("retention", "Audit logs kept 7 years")
print("after review:", trace(BRS, FRS, IRS))

> **Observe:** the first draft has no FR for BR3 and no residency or retention IR. Those are exactly the gaps that stall a project at security review weeks later. The homework's exercise 3 adds dangling references and per-FR checks.

## 3.8 A first-cut solution shape on Claude and OpenAI

Four decisions: entry point, pattern, model tier (to be settled by the eval) and context strategy. The function below encodes the README's table. [Theory](README.md#38-a-first-cut-solution-shape-on-claude-and-openai)

In [ ]:
ENTRY_POINTS = {  # entry -> (Claude, OpenAI)
    "chat_app": ("Claude Enterprise", "ChatGPT Enterprise"),
    "api": ("Messages API", "Responses API"),
    "embedded_agent": ("Claude Agent SDK", "OpenAI Agents SDK"),
    "dev_tooling": ("Claude Code", "Codex"),
    "tool_server": ("MCP server", "MCP server (remote MCP tool)"),
}
MODEL_CANDIDATES = {"claude": ["claude-haiku-4-5", "claude-sonnet-5-5"], "openai": ["gpt-6-luna", "gpt-6.1-sol"]}


def solution_shape(p: dict) -> dict:
    if p["users"] == "developers":
        entry = "dev_tooling"
    elif p["shared_by_many_clients"]:
        entry = "tool_server"
    elif not p["existing_system"]:
        entry = "chat_app"
    elif p["open_ended"]:
        entry = "embedded_agent"
    else:
        entry = "api"
    pattern = "agent" if p["open_ended"] else ("workflow" if p["steps_known"] else "augmented call")
    context = [c for c, need in (("tools", p["live_lookup"]), ("retrieval", p["documents"])) if need]
    claude, oai = ENTRY_POINTS[entry]
    return {"entry": entry, "claude": claude, "openai": oai, "pattern": pattern,
            "context": context or ["prompt only"], "models_to_eval": MODEL_CANDIDATES}


HARBOR_PROFILE = dict(users="adjusters", shared_by_many_clients=False, existing_system=True, open_ended=False,
                      steps_known=True, live_lookup=True, documents=False)
ANALYSTS_PROFILE = dict(users="analysts", shared_by_many_clients=False, existing_system=False, open_ended=False,
                        steps_known=False, live_lookup=False, documents=True)
for name, prof in (("Harbor Mutual intake", HARBOR_PROFILE), ("Policy analysts' memos", ANALYSTS_PROFILE)):
    print(name, "->", json.dumps(solution_shape(prof), indent=None))

## 3.9 Baselines, the value case and ROI

The run cost per task comes from tokens and prices (per million tokens, as used in Claude [07 §7.9](../../anthropic-claude/07-message-batches/README.md#79-cost-modeling-batch-discount-prompt-caching-and-model-choice) and OpenAI [07 §7.10](../../openai-codex/07-batch-and-cost/README.md#710-cost-modeling-per-request-and-per-workload); re-check the pricing pages). Then the README's `value_case`, three scenarios and a sensitivity check. [Theory](README.md#39-baselines-the-value-case-and-roi)

In [ ]:
PRICES = {"claude-haiku-4-5": (1.00, 5.00), "claude-sonnet-5-5": (2.00, 10.00), "gpt-6-luna": (0.10, 0.50)}


def token_cost(calls: list[tuple[str, int, int]], overhead: float = 1.2) -> float:
    # calls = [(model, input_tokens, output_tokens), ...] per task; overhead covers retries and tests.
    return overhead * sum((i * PRICES[m][0] + o * PRICES[m][1]) / 1e6 for m, i, o in calls)


RUN_COST = {"claude": token_cost([("claude-haiku-4-5", 1500, 50), ("claude-sonnet-5-5", 3000, 600)]),
            "openai": token_cost([("gpt-6-luna", 1500, 50), ("gpt-6-luna", 3000, 600)])}
print({k: f"${v:.4f} per claim" for k, v in RUN_COST.items()})


def value_case(volume, min_before, min_after, rate_per_hour, adoption, run_cost_per_task, build_cost, yearly_fixed_cost):
    hours_saved = volume * adoption * (min_before - min_after) / 60
    benefit = hours_saved * rate_per_hour
    yearly_cost = volume * adoption * run_cost_per_task + yearly_fixed_cost
    net = benefit - yearly_cost
    total_cost = build_cost + yearly_cost
    return {"benefit": round(benefit), "total_cost": round(total_cost),
            "roi_year1": round((net - build_cost) / total_cost, 2),
            "payback_months": round(build_cost / (net / 12), 1) if net > 0 else None}


BASE = dict(volume=208_000, min_before=18, min_after=6, rate_per_hour=55, adoption=0.7,
            run_cost_per_task=0.02, build_cost=300_000, yearly_fixed_cost=150_000)
assert value_case(**BASE) == {"benefit": 1601600, "total_cost": 452912, "roi_year1": 2.54, "payback_months": 2.5}

SCENARIOS = {"pessimistic": dict(adoption=0.4, min_after=10), "expected": {}, "optimistic": dict(adoption=0.9, min_after=5)}
for name, change in SCENARIOS.items():
    for vendor, cost in RUN_COST.items():
        print(f"{name:12} {vendor:7}", value_case(**{**BASE, **change, "run_cost_per_task": cost}))

In [ ]:
# Sensitivity: move each input by +/-20% and see how far year-1 ROI swings.
def swing(key: str, delta: float = 0.2) -> float:
    lo = value_case(**{**BASE, key: BASE[key] * (1 - delta)})["roi_year1"]
    hi = value_case(**{**BASE, key: BASE[key] * (1 + delta)})["roi_year1"]
    return round(abs(hi - lo), 2)


for key in ("adoption", "min_after", "rate_per_hour", "run_cost_per_task", "yearly_fixed_cost", "build_cost"):
    print(f"{key:18} ROI swing {swing(key):5}")

> **Observe:** choosing `gpt-6-luna` over the Claude pair cuts the run cost per claim by more than 10×, yet year-1 ROI barely moves, because tokens are under 2% of the yearly cost. Adoption and the loaded hourly rate move ROI the most, then build cost and minutes saved; the run cost per task barely registers. That is why the baseline, the adoption test (3.1) and finance's agreed hourly rate matter more than the price list, and why the model is chosen by the eval, not by price.

## 3.10 Success criteria, pilot design and the evaluation plan

Criteria agreed before the build. A missed guardrail means no-go; a missed target with green guardrails means iterate; a metric that was not measured counts as not passed. [Theory](README.md#310-success-criteria-pilot-design-and-the-evaluation-plan)

In [ ]:
CRITERIA = [  # (metric, op, threshold, kind)
    ("median_hours_to_claim", "<=", 2, "target"),
    ("policy_number_accuracy", ">=", 0.98, "quality"),
    ("loss_type_accuracy", ">=", 0.92, "quality"),
    ("wrong_policy_auto_created", "<=", 0, "guardrail"),
    ("p95_seconds_per_email", "<=", 90, "guardrail"),
    ("cost_per_claim_usd", "<=", 0.05, "guardrail"),
    ("share_routed_to_review", "<=", 0.25, "guardrail"),
]


def decide(results: dict, criteria: list[tuple]) -> str:
    def passed(metric, op, limit):
        v = results.get(metric)
        return v is not None and (v <= limit if op == "<=" else v >= limit)
    failed = [(m, kind) for m, op, limit, kind in criteria if not passed(m, op, limit)]
    if any(kind == "guardrail" for _, kind in failed):
        return "no-go"
    return "iterate" if failed else "go"


PILOTS = {
    "clean": dict(median_hours_to_claim=1.4, policy_number_accuracy=0.985, loss_type_accuracy=0.94,
                  wrong_policy_auto_created=0, p95_seconds_per_email=40, cost_per_claim_usd=0.02,
                  share_routed_to_review=0.18),
    "slow target": dict(median_hours_to_claim=3.1, policy_number_accuracy=0.985, loss_type_accuracy=0.94,
                        wrong_policy_auto_created=0, p95_seconds_per_email=40, cost_per_claim_usd=0.02,
                        share_routed_to_review=0.18),
    "one bad claim": dict(median_hours_to_claim=1.2, policy_number_accuracy=0.99, loss_type_accuracy=0.95,
                          wrong_policy_auto_created=1, p95_seconds_per_email=38, cost_per_claim_usd=0.02,
                          share_routed_to_review=0.15),
    "review share not logged": dict(median_hours_to_claim=1.3, policy_number_accuracy=0.99, loss_type_accuracy=0.95,
                                    wrong_policy_auto_created=0, p95_seconds_per_email=38, cost_per_claim_usd=0.02),
}
for name, results in PILOTS.items():
    print(f"{name:24} -> {decide(results, CRITERIA)}")

> **Observe:** "one bad claim" has the best business number and is still a **no-go**: a guardrail is not traded against a KPI. "Review share not logged" is also a no-go, because an unmeasured guardrail is not a passed one. Plan the logging in the eval plan, not after the pilot.

## 3.11 Domain fluency: what changes by industry

The method stays the same; the industry adds questions and infrastructure requirements. A small checklist turns the README's table into extra IR rows. [Theory](README.md#311-domain-fluency-what-changes-by-industry)

In [ ]:
DOMAIN_CHECKS = {  # industry -> [(question to ask early, IR category, IR text)]
    "insurance": [("Which decisions must stay with a licensed person?", "approval", "Licensed adjuster signs off on claim decisions")],
    "healthcare": [("Is PHI in scope, and which platforms are covered by a BAA?", "compliance", "Platform covered by a signed BAA"),
                   ("Where may PHI be processed?", "residency", "Processing in the approved region only")],
    "financial_services": [("Does this count as a model under the model-risk policy?", "compliance", "Model inventory entry and validation")],
    "public_sector": [("Which authorization level and region are required?", "compliance", "Authorized platform (for example FedRAMP)")],
}


def add_domain_requirements(irs: dict, industry: str) -> dict:
    extra = {f"IR-D{i}": (cat, text) for i, (_, cat, text) in enumerate(DOMAIN_CHECKS.get(industry, []), 1)}
    return {**irs, **extra}


for q, _, _ in DOMAIN_CHECKS["healthcare"]:
    print("ask:", q)
print(add_domain_requirements({"IR1": ("throughput", "1,300 faxes/week")}, "healthcare"))

## Mini-project: a one-page scope brief for Brightline Clinics

**Brief (fictional).** Brightline Clinics, a 30-site outpatient network, receives about **1,250 referral faxes a week**. Intake coordinators read each fax, find or create the patient, pick the specialty and enter the referral. The operations director asks for "something with AI for referrals".

Build `build_scope_brief()` from the functions above:

1. Structure the interview notes with **both** vendors and keep only supported facts.
2. Prioritize the candidate use cases and pick the top quick win.
3. Lint the problem statement and trace the requirements (with the healthcare checks added).
4. Sketch the solution shape for Claude and OpenAI.
5. Compute the value case in three scenarios.
6. Print a one-page Markdown brief with the decision rule and the open assumptions.

In [ ]:
BL_TRANSCRIPT = '''Architect: Walk me through the last referral fax.
Priya (Brightline intake manager): A cardiology referral came in this morning, three pages, handwritten notes on page two.
I matched the patient by name and date of birth, picked cardiology and typed it into our scheduling system.
Architect: How many a week?
Priya: The fax server log shows 1,250 a week on average.
Architect: How long does each take?
Priya: We timed it last month: 14 minutes per referral.
Architect: What goes wrong?
Priya: Wrong specialty about 4% of the time, according to our audit, and then the patient waits another week.
Architect: Constraints?
Priya: It's patient data, so it has to stay with platforms covered by our BAA.'''

BL_NOTES = {"facts": [
    f("volume", "About 1,250 referral faxes a week", "The fax server log shows 1,250 a week on average", True),
    f("workflow", "14 minutes per referral", "We timed it last month: 14 minutes per referral", True),
    f("pain", "Wrong specialty about 4% of the time", "Wrong specialty about 4% of the time, according to our audit", True),
    f("constraint", "Only BAA-covered platforms may process PHI", "it has to stay with platforms covered by our BAA", True),
], "open_questions": ["Share of faxes with handwriting?", "Does the scheduling system have an API?"]}

BL_USE_CASES = [
    uc("Referral intake: classify specialty, extract patient and referrer", value=(4, 4, 4), feas=(4, 4, 3, 4)),
    uc("Auto-schedule appointments", gates=(1, 1, 1, 0), value=(4, 3, 4), feas=(2, 3, 2, 3)),
    uc("Patient FAQ assistant", value=(2, 2, 3), feas=(4, 4, 4, 3)),
]
BL_PS = ProblemStatement(
    who="Referral intake coordinators (9 people)", job="Turn a referral fax into a correctly routed referral",
    baseline="14 min per referral (timed last month); 4% wrong specialty (audit)",
    target="5 min per referral; wrong specialty at or below 2%", constraints=["PHI on BAA-covered platforms only"],
    out_of_scope=["Scheduling", "Clinical triage"], decision_by="Day 30 of the pilot, on the agreed success criteria")
BL_BRS = {"BR1": "Referrals routed the same day", "BR2": "Fewer wrong-specialty referrals"}
BL_FRS = {"FR1": ("Classify the specialty", ["BR1", "BR2"]), "FR2": ("Extract patient and referrer fields", ["BR1"]),
          "FR3": ("Route low-confidence referrals to a coordinator", ["BR2"])}
BL_IRS = {"IR1": ("throughput", "Peak 60 faxes/hour"), "IR2": ("latency", "p95 under 2 min"),
          "IR3": ("integration", "Scheduling system API"), "IR4": ("identity", "SSO for coordinators"),
          "IR5": ("retention", "Audit log per referral, 6 years")}
BL_VALUE = dict(volume=65_000, min_before=14, min_after=5, rate_per_hour=38, adoption=0.8,
                run_cost_per_task=0.02, build_cost=180_000, yearly_fixed_cost=90_000)
BL_PROFILE = dict(users="coordinators", shared_by_many_clients=False, existing_system=True, open_ended=False,
                  steps_known=True, live_lookup=True, documents=False)

In [ ]:
def build_scope_brief(name, transcript, notes_script, use_cases, ps, brs, frs, irs, industry, profile, value):
    llm = FakeLLM(lambda path, body: notes_script)
    facts = []
    for vendor in ("claude", "openai"):
        notes = structured(vendor, NOTES_SYSTEM, transcript, DiscoveryNotes, llm.clients)
        facts = [x for x in notes.facts if x not in unsupported(notes, transcript)]
    ranked = sorted(use_cases, key=lambda u: (not prioritize(u)[0].startswith("quick win"), -prioritize(u)[1]))
    top, parked = ranked[0], [u for u in ranked if prioritize(u)[0].startswith("park")]
    irs = add_domain_requirements(irs, industry)
    gaps, issues, shape = trace(brs, frs, irs), lint(ps), solution_shape(profile)
    rows = [f"| {s} | {r['benefit']:,} | {r['total_cost']:,} | {r['roi_year1']} | {r['payback_months']} |"
            for s, r in ((s, value_case(**{**value, **c})) for s, c in SCENARIOS.items())]
    lines = [f"# Scope brief: {name}", "", "## Problem", f"**{ps.who}** need to: {ps.job}.",
             f"Baseline: {ps.baseline}. Target: {ps.target}. Out of scope: {', '.join(ps.out_of_scope)}.",
             f"Problem-statement lint: {issues or 'clean'}", "", "## Use case",
             f"Pilot first: **{top['name']}** ({prioritize(top)[0]}, value {prioritize(top)[1]}, feasibility {prioritize(top)[2]}).",
             *[f"- Parked: {u['name']} ({prioritize(u)[0]})" for u in parked], "", "## Requirements",
             f"{len(brs)} BR, {len(frs)} FR, {len(irs)} IR. Trace gaps: {gaps}", "", "## Solution shape",
             f"| Decision | Claude | OpenAI |", "|---|---|---|", f"| Entry point | {shape['claude']} | {shape['openai']} |",
             f"| Pattern | {shape['pattern']} | {shape['pattern']} |",
             f"| Models to evaluate | {', '.join(MODEL_CANDIDATES['claude'])} | {', '.join(MODEL_CANDIDATES['openai'])} |",
             f"| Context | {', '.join(shape['context'])} | {', '.join(shape['context'])} |", "", "## Value case (year 1)",
             "| Scenario | Benefit $ | Total cost $ | ROI | Payback (months) |", "|---|---|---|---|---|", *rows, "",
             "## Evidence and open questions", *[f"- {'confirmed' if x.confirmed else 'ASSUMPTION'}: {x.statement}" for x in facts],
             *[f"- open: {q}" for q in notes.open_questions], "", "## Decision rule",
             "Go only if every guardrail passes; iterate if a target misses; decided on day 30 of the pilot."]
    return "\n".join(lines), gaps, issues


brief, gaps, issues = build_scope_brief("Brightline Clinics", BL_TRANSCRIPT, BL_NOTES, BL_USE_CASES, BL_PS, BL_BRS,
                                        BL_FRS, BL_IRS, "healthcare", BL_PROFILE, BL_VALUE)
print(brief)
assert not issues and not gaps["br_without_fr"] and not gaps["missing_ir_categories"], (issues, gaps)
for header in ("## Problem", "## Use case", "## Requirements", "## Solution shape", "## Value case", "## Decision rule"):
    assert header in brief, header

> **Observe:** the expected case pays back in about 10 months, but the pessimistic case is negative. The honest brief says so, and it names the input the decision depends on (adoption), so the pilot is designed to measure it. Compare with Harbor Mutual, where every scenario is positive: same method, different answer.

In [ ]:
# (live, costs money: about $0.01) structure the Brightline interview with the real APIs and check the quotes
if not RUN_LIVE:
    print("Skipped: set RUN_LIVE=1 plus ANTHROPIC_API_KEY and OPENAI_API_KEY in .env to run this cell.")
else:
    live = {"claude": anthropic.Anthropic(), "openai": openai.OpenAI()}
    for vendor in ("claude", "openai"):
        notes = structured(vendor, NOTES_SYSTEM, BL_TRANSCRIPT, DiscoveryNotes, live)
        print(vendor, "| facts:", len(notes.facts), "| unsupported:", [x.quote for x in unsupported(notes, BL_TRANSCRIPT)])

## Recap

- **Discovery is a chain of artifacts**, and any link can end the project early. Keep an assumptions log and test the riskiest one first.
- **Let a model structure notes, never supply facts.** Quote-check in code; unconfirmed statements become assumptions.
- **Walk the workflow, then tag steps.** A large share of the value is often plain integration code.
- **Gate, then score.** No data, no owner, no measurement or no safe failure means park, whatever the value.
- **Name the gain.** Structural gains show up in a 30-day pilot; marginal gains rarely do.
- **Problem → FR → IR.** Missing residency or retention requirements are what stall security review.
- **Tokens are rarely the cost that matters.** Adoption and minutes saved drive ROI; show three scenarios.
- **Guardrails are not traded for KPIs.** Unmeasured counts as failed.

Next: [homework](02_homework.ipynb) · Back to the [chapter README](README.md)